In [ ]:
import pandas as pd
import numpy as np

# Load factor library
factor_file = "data/processed/factor_library.parquet"
df = pd.read_parquet(factor_file)

# Inspect the data
df.head()


In [ ]:
# Example: pivot each factor into a wide table per ticker
pivot_factors = df.pivot_table(
    index='date',
    columns='ticker',
    values='momentum_12m'  # choose a factor or combine multiple later
)

pivot_factors.head()


In [ ]:
# Compute correlation matrix between tickers
corr_matrix = pivot_factors.corr()
corr_matrix.head()


In [ ]:
# Simple adjacency: use correlations as weights
A = corr_matrix.values  # adjacency matrix

# Optionally threshold to remove weak correlations
threshold = 0.3
A[A < threshold] = 0
A


In [ ]:
np.save("data/processed/graph_adj.npy", A)
import matplotlib.pyplot as plt
print("✅ Saved adjacency matrix to data/processed/graph_adj.npy")
Image = np.load("data/processed/graph_adj.npy")

# Show as an image
plt.imshow(Image, cmap='viridis')
plt.colorbar(label="Weight")
plt.title("Graph Adjacency Matrix")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import os

# Paths
factor_file = "data/processed/factor_library.parquet"
sector_file = "data/raw/ticker_sector.csv"  # CSV mapping tickers -> sectors
save_path = "data/processed"
os.makedirs(save_path, exist_ok=True)

print("✅ Libraries loaded and paths set")


In [ ]:
# Load factor library
df = pd.read_parquet(factor_file)
df.head()


In [ ]:
# Example ticker-sector CSV:
# ticker,sector
# AAPL,Tech
# MSFT,Tech
# AMZN,Consumer
# GOOG,Tech

sectors = pd.read_csv(sector_file)
sectors.set_index('ticker', inplace=True)
sectors.head()


In [ ]:
# Choose a factor (e.g., momentum_12m)
factor = 'momentum_12m'

pivot_factors = df.pivot_table(
    index='date',
    columns='ticker',
    values=factor
)

# Optional: forward-fill missing values
pivot_factors = pivot_factors.fillna(method='ffill')
pivot_factors.head()


In [ ]:
# Pairwise correlation between tickers
corr_matrix = pivot_factors.corr()
corr_matrix.head()


In [ ]:
# Sector-based adjacency enhancement
adj = corr_matrix.copy()

# Boost correlation for same-sector tickers
for t1 in adj.columns:
    for t2 in adj.columns:
        if t1 == t2:
            continue
        if sectors.loc[t1, 'sector'] == sectors.loc[t2, 'sector']:
            adj.loc[t1, t2] += 0.1  # boost weight for same sector

# Clip max correlation to 1
adj = adj.clip(upper=1.0)

adj.head()


In [ ]:

# Sector-based adjacency enhancement
adj = corr_matrix.copy()

# Boost correlation for same-sector tickers
for t1 in adj.columns:
    for t2 in adj.columns:
        if t1 == t2:
            continue
        if sectors.loc[t1, 'sector'] == sectors.loc[t2, 'sector']:
            adj.loc[t1, t2] += 0.1  # boost weight for same sector

# Clip max correlation to 1
adj = adj.clip(upper=1.0)

adj.head()
threshold = 0.3
adj_thresh = adj.where(adj >= threshold, 0)
adj_thresh.head()

adj_path = os.path.join(save_path, "graph_adj.npy")
np.save(adj_path, adj_thresh.values)
print(f"✅ Saved adjacency matrix to {adj_path}")
